In [1]:
import os
import pandas as pd
import numpy as np

DADOS = '../dados'
OUTPUT = '../output'

In [2]:
df_sih_geral = pd.read_csv(f'{DADOS}/dados_sih_2024.csv', low_memory=False)
cnes_mapping = pd.read_csv(f'{DADOS}/CNESBR.csv')
df_porte = pd.read_csv(f'{DADOS}/cnes_porte_CE_2024.csv')
geo_data = pd.read_csv(f'{DADOS}/geograf.csv')

print(df_sih_geral.shape)
print(df_sih_geral.columns)

(587132, 113)
Index(['UF_ZI', 'ANO_CMPT', 'MES_CMPT', 'ESPEC', 'CGC_HOSP', 'N_AIH', 'IDENT',
       'CEP', 'MUNIC_RES', 'NASC',
       ...
       'DIAGSEC9', 'TPDISEC1', 'TPDISEC2', 'TPDISEC3', 'TPDISEC4', 'TPDISEC5',
       'TPDISEC6', 'TPDISEC7', 'TPDISEC8', 'TPDISEC9'],
      dtype='object', length=113)


In [3]:
def padronizar_cnes_coluna(df, coluna='CNES'):
    df[coluna] = (
        df[coluna]
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.strip()
        .str.zfill(7)
    )
    return df

df_sih_geral = padronizar_cnes_coluna(df_sih_geral, 'CNES')
cnes_mapping = padronizar_cnes_coluna(cnes_mapping, 'CNES')
df_porte = padronizar_cnes_coluna(df_porte, 'CNES')

In [4]:
df_hospitais_ativos = df_sih_geral[['CNES']].drop_duplicates()

df_hospitais_ativos = df_hospitais_ativos.merge(
    cnes_mapping.rename(columns={'NOMEFANT': 'HOSPITAL'})[['CNES', 'HOSPITAL']],
    on='CNES',
    how='left'
)

print(df_hospitais_ativos.head())
print(f'Total de hospitais ativos em 2024: {len(df_hospitais_ativos)}')
print(f"Hospitais sem nome: {df_hospitais_ativos['HOSPITAL'].isna().sum()}")

      CNES                                  HOSPITAL
0  9672427       HOSPITAL REGIONAL VALE DO JAGUARIBE
1  2561417  HOSPITAL SAO JOSE DE DOENCAS INFECCIOSAS
2  2480026     HOSPITAL DE SAUDE MENTAL DE MESSEJANA
3  7061021       HOSPITAL REGIONAL DO SERTAO CENTRAL
4  6848710                   HOSPITAL REGIONAL NORTE
Total de hospitais ativos em 2024: 238
Hospitais sem nome: 4


In [5]:
df_hospitais_ativos = df_hospitais_ativos.merge(
    df_porte[['CNES', 'PORTE']],
    on='CNES',
    how='left'
)

df_hospitais_ativos['PORTE'] = df_hospitais_ativos['PORTE'].fillna('Desconhecido')

print(df_hospitais_ativos['PORTE'].value_counts())

PORTE
Pequeno     138
Medio        76
Grande       22
Especial      2
Name: count, dtype: int64


In [6]:
from normalize import normalizar_nome

df_hospitais_ativos['NOME_NORM'] = df_hospitais_ativos['HOSPITAL'].apply(normalizar_nome)
geo_data['NOME_NORM'] = geo_data['Nome'].apply(normalizar_nome)

df_hospitais_ativos = df_hospitais_ativos.merge(
    geo_data[['NOME_NORM', 'Latitude', 'Longitude']],
    on='NOME_NORM',
    how='left'
)

In [ ]:
df_hospitais_ativos

In [7]:
sem_coord = df_hospitais_ativos[
    df_hospitais_ativos['Latitude'].isna() |
    df_hospitais_ativos['Longitude'].isna()
]

print(f'Hospitais sem coordenada: {len(sem_coord)}')
print(sem_coord[['CNES', 'HOSPITAL', 'PORTE']].head(30))

Hospitais sem coordenada: 66
       CNES                                          HOSPITAL    PORTE
2   2480026             HOSPITAL DE SAUDE MENTAL DE MESSEJANA   Grande
8   3242587                          HOSPITAL REGIONAL UNIMED  Pequeno
14  2527057          CASA DE SAUDE E MATERNIDADE SAO RAIMUNDO  Pequeno
21  2563347                  HOSPITAL ANTONIO ROSENO DE MATOS  Pequeno
25  2373009    HOSPITAL E MATERNIDADE SANTA LUISA DE MARILLAC    Medio
28  2372487                 HOSP MATERN LIA LOIOLA DE ALENCAR  Pequeno
30  2328038                    HOSPITAL MUNICIPAL DE ARNEIROZ  Pequeno
33  2552345                            HOSPITAL SAO FRANCISCO  Pequeno
38  2373475                              HOSPITAL DE BARREIRA  Pequeno
45  2611635                                             INCRI    Medio
50  2561034    HOSPITAL E MATERNIDADE NOSSA SENHORA DE NAZARE  Pequeno
53  2425343                     HOSP MATERN GERALDO L BOTELHO  Pequeno
57  2333864            HOSPITAL MUNICIPAL DR GEN

In [ ]:
sem_coord

In [8]:
sem_coord_export = sem_coord[['CNES', 'HOSPITAL', 'Latitude', 'Longitude', 'PORTE']].copy()

sem_coord_export.to_csv(
    f'{DADOS}/hospitais_sem_coordenada.csv',
    index=False,
    encoding='utf-8-sig'
)

print(f'Arquivo exportado com {len(sem_coord_export)} hospitais sem coordenada.')

Arquivo exportado com 66 hospitais sem coordenada.


In [10]:
df_coord_manual = pd.read_csv(
    f'{DADOS}/hospitais_sem_coordenada_preenchido.csv',
    sep=';',
    encoding='utf-8-sig'
)

df_coord_manual.head()

df_coord_manual = df_coord_manual.dropna(how='all').copy()

df_coord_manual = df_coord_manual[
    df_coord_manual['CNES'].notna()
].copy()

In [11]:
df_coord_manual = pd.read_csv(
    f'{DADOS}/hospitais_sem_coordenada_preenchido.csv',
    sep=';',
    encoding='utf-8-sig'
)

df_coord_manual['CNES'] = (
    df_coord_manual['CNES']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
    .str.zfill(7)
)

df_coord_manual['HOSPITAL'] = df_coord_manual['HOSPITAL'].astype(str).str.strip()

df_coord_manual['Latitude'] = pd.to_numeric(
    df_coord_manual['Latitude'],
    errors='coerce'
)

df_coord_manual['Longitude'] = pd.to_numeric(
    df_coord_manual['Longitude'],
    errors='coerce'
)

df_coord_manual['PORTE'] = (
    df_coord_manual['PORTE']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)

In [12]:
print('Registros no arquivo manual:', len(df_coord_manual))
print('Sem latitude:', df_coord_manual['Latitude'].isna().sum())
print('Sem longitude:', df_coord_manual['Longitude'].isna().sum())

df_coord_manual[
    df_coord_manual['Latitude'].isna() |
    df_coord_manual['Longitude'].isna()
]

Registros no arquivo manual: 66
Sem latitude: 0
Sem longitude: 0


,CNES,HOSPITAL,Latitude,Longitude,PORTE


In [13]:
fora_ceara = df_coord_manual[
    (df_coord_manual['Latitude'] < -8.5) |
    (df_coord_manual['Latitude'] > -2.0) |
    (df_coord_manual['Longitude'] < -42.5) |
    (df_coord_manual['Longitude'] > -37.0)
]

print(f'Coordenadas fora do intervalo esperado do Ceará: {len(fora_ceara)}')
fora_ceara[['CNES', 'HOSPITAL', 'Latitude', 'Longitude', 'PORTE']]

Coordenadas fora do intervalo esperado do Ceará: 0


,CNES,HOSPITAL,Latitude,Longitude,PORTE


In [14]:
df_hospitais_ativos = df_hospitais_ativos.merge(
    df_coord_manual[['CNES', 'HOSPITAL', 'Latitude', 'Longitude']],
    on='CNES',
    how='left',
    suffixes=('', '_MANUAL')
)

df_hospitais_ativos['HOSPITAL'] = df_hospitais_ativos['HOSPITAL'].fillna(
    df_hospitais_ativos['HOSPITAL_MANUAL']
)

df_hospitais_ativos['Latitude'] = df_hospitais_ativos['Latitude'].fillna(
    df_hospitais_ativos['Latitude_MANUAL']
)

df_hospitais_ativos['Longitude'] = df_hospitais_ativos['Longitude'].fillna(
    df_hospitais_ativos['Longitude_MANUAL']
)

df_hospitais_ativos = df_hospitais_ativos.drop(
    columns=['HOSPITAL_MANUAL', 'Latitude_MANUAL', 'Longitude_MANUAL'],
    errors='ignore'
)

In [15]:
print('Hospitais ativos:', len(df_hospitais_ativos))
print('Sem nome:', df_hospitais_ativos['HOSPITAL'].isna().sum())
print('Sem latitude:', df_hospitais_ativos['Latitude'].isna().sum())
print('Sem longitude:', df_hospitais_ativos['Longitude'].isna().sum())

print(df_hospitais_ativos['PORTE'].value_counts())

Hospitais ativos: 238
Sem nome: 0
Sem latitude: 0
Sem longitude: 0
PORTE
Pequeno     138
Medio        76
Grande       22
Especial      2
Name: count, dtype: int64


In [16]:
df_hospitais_ativos.to_csv(
    f'{DADOS}/hospitais_ativos_CE_2024_geocodificados.csv',
    index=False,
    encoding='utf-8-sig'
)

print('Base final salva com sucesso.')

Base final salva com sucesso.


In [ ]:
df_hospitais_ativos